
# 02_SOLVER - KRONA LTP

Solver oficial de capacidade para o projeto KRONA/LTP.

Esta versão mantém o padrão do **V1 por rateio proporcional com rastreabilidade**, porém incorpora a regra nova de **dupla restrição de capacidade**:

- limite mensal do **recurso produtivo** (`ALOC_REC`);
- limite mensal da **ferramenta** (`COD_FER_UNID`).

O versionamento do notebook é feito pelo **Git**. Por isso, o arquivo oficial permanece sem sufixo de versão.

- Notebook: `02_SOLVER.ipynb`
- Entrada: `bd_LTP_NEC_SOLVER.xlsx`
- Saída: `bd_SOLVER_v02.xlsx`

---

## Premissas desta versão

- Processa somente o primeiro `MES_REF` da base.
- A demanda do item é `NEC_PCS`.
- `NEC_PCS` pertence ao item no mês, no nível `MES_REF + ID_PROD_UNID_FAT`.
- `NEC_PCS` não pertence ao recurso, ferramenta ou roteiro.
- `PCS_HORA` pertence ao roteiro/recurso/ferramenta e só é usado quando a demanda em peças é convertida em horas.
- Roteiros e alternativas são ordenados por `PRIOR_MATPAR` e `PRIOR_ROT`.
- Menor prioridade é melhor.
- Alternativas são permitidas; o saldo não atendido pode seguir para alternativas posteriores.
- Não existe faixa, meta ou percentual mínimo artificial.
- O corte de capacidade é feito por **rateio proporcional do estouro**.

---

## Regra de capacidade

O modelo controla capacidade em dois níveis ao mesmo tempo.

### 1. Capacidade do recurso produtivo

```text
SOMA(HR_PRODUZIR_SOLVER) por ALOC_REC <= HOR_REC
```

### 2. Capacidade mensal da ferramenta

```text
SOMA(HR_PRODUZIR_SOLVER) por COD_FER_UNID <= HOR_FER
```

Essa regra é necessária porque uma mesma ferramenta pode aparecer em mais de um recurso. Mesmo que exista capacidade em mais de uma máquina, a ferramenta não pode ultrapassar sua capacidade mensal total.

O caso que motivou esta correção foi:

```text
COD_FER_UNID = 0647B|MAT
```

---

## Regra do HOR_CAP

A capacidade oficial da alternativa é:

```text
HOR_CAP = min(HOR_REC, HOR_FER)
```

Sem fallback.

Se `HOR_REC = 0` ou `HOR_FER = 0`, então `HOR_CAP = 0`.

Alternativas com `HOR_CAP <= 0` não entram no plano de produção, mas permanecem nas guias de auditoria e diagnóstico.

Importante: `HOR_CAP` continua sendo mantido para auditoria da alternativa. Porém o motor também controla os saldos acumulados de `ALOC_REC` e `COD_FER_UNID` durante a alocação.

---

## Regra de rateio proporcional

Em cada rodada de prioridade, o motor pega os itens com saldo pendente e tenta alocá-los nas alternativas válidas daquele nível.

Para cada tentativa, a capacidade disponível é o menor saldo entre recurso e ferramenta:

```text
CAP_DISPONIVEL_LINHA = min(CAP_RESTANTE_ALOC_REC, CAP_RESTANTE_COD_FER_UNID)
```

Quando a demanda em horas ultrapassa a capacidade disponível, o corte é proporcional:

```text
FATOR_RATEIO = CAP_DISPONIVEL_LINHA / HR_DEMANDA_TOTAL_LIMITANTE
```

Assim, os itens que disputam o mesmo gargalo pagam proporcionalmente o estouro.

Exemplo:

```text
Demanda total = 600 h
Capacidade disponível = 500 h
Fator = 500 / 600 = 0,8333
```

Leitura:

```text
Atende 83,33%
Corta 16,67%
```

---

## Ordem de processamento

A ordem das alternativas respeita:

```text
PRIOR_MATPAR crescente
PRIOR_ROT crescente
```

Exemplo:

```text
1º PRIOR_MATPAR = 1 / PRIOR_ROT = 1
2º PRIOR_MATPAR = 1 / PRIOR_ROT = 2
3º PRIOR_MATPAR = 2 / PRIOR_ROT = 1
4º PRIOR_MATPAR = 2 / PRIOR_ROT = 2
```

O saldo não atendido de uma rodada segue para a próxima alternativa válida do item.

---

## Alternativas inválidas

Uma alternativa não entra no plano se tiver:

```text
HOR_REC <= 0
HOR_FER <= 0
HOR_CAP <= 0
PCS_HORA <= 0
```

Mesmo inválida, ela permanece nas guias de auditoria e diagnóstico, para explicar se o item não tinha alternativa real, se tinha alternativa sem capacidade, ou se tinha produtividade inválida.

---

## Não atendimento

O não atendimento final é controlado em peças:

```text
QTD_NAO_ATEND_SOLVER
```

Não calcular gap final em horas por item, porque `PCS_HORA` pertence à alternativa produtiva, não ao produto. Se o item não foi atendido, ele não está associado a uma produtividade única.

---

## Rastreabilidade inteligente

O modelo gera logs para responder perguntas como:

- Por que este item produziu só em uma máquina?
- Por que o saldo não foi para outra alternativa?
- Quem consumiu a capacidade antes?
- O corte aconteceu por falta de recurso ou por falta de ferramenta?
- A alternativa estava inválida ou apenas sem saldo disponível?

A guia principal para isso é:

```text
07_RASTRO_RATEIO
```

Ela registra, por tentativa:

- rodada de prioridade;
- item;
- recurso;
- ferramenta;
- saldo pendente antes;
- demanda em horas;
- saldo restante do recurso;
- saldo restante da ferramenta;
- capacidade disponível da linha;
- fator de rateio;
- motivo limitante;
- quantidade produzida;
- saldo pendente depois;
- status da decisão.

---

## Saídas

A planilha final deve conter:

1. `01_RESUMO_RECURSOS`
2. `02_RESUMO_FERRAMENTAS`
3. `03_PLANO_PRODUCAO`
4. `04_NAO_ATEND_ITEM`
5. `05_AUDITORIA_CAPACIDADE`
6. `06_ALTERNATIVAS_ITEM`
7. `07_RASTRO_RATEIO`
8. `08_DIAGNOSTICO_ITEM`
9. `09_DIAGNOSTICO_ALTERNATIVAS`
10. `99_VALIDACAO`

---

## Validações obrigatórias

O notebook precisa validar:

```text
SOMA(QTD_ATENDIDA_SOLVER + QTD_NAO_ATEND_SOLVER) = SOMA(NEC_PCS)
```

```text
SOMA(HR_PRODUZIR_SOLVER) por ALOC_REC <= HOR_REC
```

```text
SOMA(HR_PRODUZIR_SOLVER) por COD_FER_UNID <= HOR_FER
```

Todas as validações ficam registradas na guia `99_VALIDACAO`.

---

## Regra central do modelo

O solver deve fazer **rateio proporcional do estouro**, respeitando simultaneamente a capacidade mensal do **recurso produtivo** e a capacidade mensal da **ferramenta**, sem perder a rastreabilidade do modelo V1.



In [ ]:

# ============================================================
# 00. CONFIGURAÇÃO
# ============================================================

print("02_SOLVER_RATEIO_PROPORCIONAL_DUPLA_RESTRICAO")

from functions import *
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
try:
    from IPython.display import display
except Exception:
    display = print
    
# Iniciando Temporizador
timer = Temporizador()
timer.iniciar()

# Caminho padrão do projeto no Windows
OUTPUT_DIR_WINDOWS = Path(r"C:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT")

# Versão do pipeline em teste
PIPELINE_VERSION = "v02"
OUTPUT_SOLVER_FILENAME = f"bd_SOLVER_{PIPELINE_VERSION}.xlsx"

INPUT_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / OUTPUT_SOLVER_FILENAME

# Fallback para execução em ambiente local/sandbox
INPUT_FILE_FALLBACK = Path("/mnt/data/bd_LTP_NEC_SOLVER.xlsx")
OUTPUT_SOLVER_FILE_FALLBACK = Path("/mnt/data") / OUTPUT_SOLVER_FILENAME

if INPUT_FILE_WINDOWS.exists():
    INPUT_FILE = INPUT_FILE_WINDOWS
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_WINDOWS
else:
    INPUT_FILE = INPUT_FILE_FALLBACK
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_FALLBACK

print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_SOLVER_FILE:", OUTPUT_SOLVER_FILE)

LOTE_MIN_FLAG = True
MULTIPLO_EMB_FLAG = True
TOL = 1e-7
DATA_HORA_EXECUCAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


In [ ]:

# ============================================================
# 01. FUNÇÕES AUXILIARES
# ============================================================

def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def calcular_nec_pcs_se_necessario(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula NEC_PCS somente se a coluna não existir na base."""
    df = df.copy()
    if "NEC_PCS" in df.columns:
        df["NEC_PCS"] = to_num(df["NEC_PCS"])
        print("NEC_PCS lido da base de entrada.")
        return df

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    base_comum = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"] - df["LTP_EST_TRANS_PCS"] - df["ORI_TOT_PCS"] - df["TRIANG_TOT_PCS"]
    )
    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")
    nec = np.where(mesma_reg_nao, base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"], base_comum)
    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    nec = np.maximum(nec, df["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = (nec > 0) & (df["LOTE_MIN"] > 0)
        nec = np.where(mask_lote, np.maximum(nec, df["LOTE_MIN"]), nec)
    if MULTIPLO_EMB_FLAG:
        tipo = df.get("TIPO_PROD", "").astype(str).str.upper()
        mask_emb = (nec > 0) & (df["QTD_EMB"] > 0) & (tipo.isin(["PA", "MR"]))
        nec = np.where(mask_emb, np.ceil(nec / df["QTD_EMB"]) * df["QTD_EMB"], nec)

    df["NEC_PCS"] = pd.Series(nec, index=df.index).fillna(0)
    print("NEC_PCS calculado internamente porque a base não trouxe a coluna.")
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()
    print("MES_REF processado:", mes_ref)
    print("Linhas bd_mes:", len(bd_mes))
    return bd_mes


def classificar_motivo_invalida(row):
    motivos = []
    if pd.isna(row.get("ALOC_REC")) or str(row.get("ALOC_REC", "")).strip() == "":
        motivos.append("ALOC_REC_AUSENTE")
    if pd.isna(row.get("COD_FER_UNID")) or str(row.get("COD_FER_UNID", "")).strip() == "":
        motivos.append("COD_FER_UNID_AUSENTE")
    if row.get("PCS_HORA", 0) <= TOL:
        motivos.append("PCS_HORA_ZERO_OU_INVALIDO")
    if row.get("HOR_REC", 0) <= TOL:
        motivos.append("HOR_REC_ZERO_OU_NEGATIVO")
    if row.get("HOR_FER", 0) <= TOL:
        motivos.append("HOR_FER_ZERO_OU_NEGATIVO")
    if row.get("HOR_CAP", 0) <= TOL:
        motivos.append("HOR_CAP_ZERO_OU_NEGATIVO")
    return " | ".join(dict.fromkeys(motivos)) if motivos else "ALTERNATIVA_VALIDA"


def formatar_excel(caminho_arquivo):
    wb = load_workbook(caminho_arquivo)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    thin_gray = Side(style="thin", color="D9E2F3")
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = Border(bottom=thin_gray)
        for col_idx, column_cells in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in column_cells:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 45)
        ws.auto_filter.ref = ws.dimensions
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.00'
    wb.save(caminho_arquivo)


In [ ]:

# ============================================================
# 02. LEITURA DA BASE
# ============================================================

bd = pd.read_excel(INPUT_FILE)
print("Base lida:", bd.shape)
print("Colunas:", len(bd.columns))
bd = calcular_nec_pcs_se_necessario(bd)
bd_mes = preparar_base(bd)


In [ ]:

# ============================================================
# 03. MONTAGEM DA DEMANDA E DAS ALTERNATIVAS
# ============================================================

meta_cols = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]

bd_demanda_solver = (
    bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
    .agg({**{c: "first" for c in meta_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns}, "NEC_PCS": "max"})
)
bd_demanda_solver = bd_demanda_solver[bd_demanda_solver["NEC_PCS"] > TOL].copy()

alt_cols = ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "ID_RECURSO", "ID_FERRAMENTA"]
bd_alternativas_item = bd_mes[[c for c in alt_cols if c in bd_mes.columns]].copy()
bd_alternativas_item = bd_alternativas_item[bd_alternativas_item["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])].copy()
bd_alternativas_item = bd_alternativas_item.drop_duplicates([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_alternativas_item.columns]).reset_index(drop=True)

bd_alternativas_item["ALTERNATIVA_VALIDA"] = (
    bd_alternativas_item["ALOC_REC"].notna() & bd_alternativas_item["COD_FER_UNID"].notna()
    & (bd_alternativas_item["PCS_HORA"] > TOL) & (bd_alternativas_item["HOR_REC"] > TOL)
    & (bd_alternativas_item["HOR_FER"] > TOL) & (bd_alternativas_item["HOR_CAP"] > TOL)
)
bd_alternativas_item["MOTIVO_INVALIDA"] = bd_alternativas_item.apply(classificar_motivo_invalida, axis=1)
bd_alternativas_validas = bd_alternativas_item[bd_alternativas_item["ALTERNATIVA_VALIDA"]].copy()

bd_cap_recursos = bd_alternativas_item.dropna(subset=["ALOC_REC"]).groupby("ALOC_REC", as_index=False).agg(HOR_REC=("HOR_REC", "max"))
bd_cap_ferramentas = bd_alternativas_item.dropna(subset=["COD_FER_UNID"]).groupby("COD_FER_UNID", as_index=False).agg(HOR_FER=("HOR_FER", "max"))

print("Itens com demanda:", len(bd_demanda_solver))
print("Alternativas auditadas:", len(bd_alternativas_item))
print("Alternativas válidas para produção:", len(bd_alternativas_validas))
print("Alternativas com HOR_CAP zero:", (bd_alternativas_item["HOR_CAP"] <= TOL).sum())
print("Recursos:", len(bd_cap_recursos))
print("Ferramentas:", len(bd_cap_ferramentas))


In [ ]:

# ============================================================
# 04. SOLVER - RATEIO PROPORCIONAL COM DUPLA RESTRIÇÃO
# ============================================================

def executar_rateio_proporcional_dupla_restricao(bd_demanda, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas):
    alternativas = bd_alternativas_validas.copy()
    cap_restante_rec = bd_cap_recursos.set_index("ALOC_REC")["HOR_REC"].to_dict()
    cap_restante_fer = bd_cap_ferramentas.set_index("COD_FER_UNID")["HOR_FER"].to_dict()
    pendente = bd_demanda.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].to_dict()
    producao, rastro = [], []

    niveis_prioridade = list(alternativas[["PRIOR_MATPAR", "PRIOR_ROT"]].drop_duplicates().sort_values(["PRIOR_MATPAR", "PRIOR_ROT"]).itertuples(index=False, name=None))
    for rodada, (prior_matpar, prior_rot) in enumerate(niveis_prioridade, start=1):
        nivel = alternativas[alternativas["PRIOR_MATPAR"].eq(prior_matpar) & alternativas["PRIOR_ROT"].eq(prior_rot)].copy()
        nivel["RODADA"] = rodada
        nivel["QTD_PENDENTE_ANTES"] = nivel["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
        nivel = nivel[nivel["QTD_PENDENTE_ANTES"] > TOL].copy()
        if nivel.empty:
            continue

        qtd_alt_mesmo_nivel = nivel.groupby("ID_PROD_UNID_FAT")["ALOC_REC"].transform("count")
        nivel["QTD_ALTERNATIVAS_MESMO_NIVEL"] = qtd_alt_mesmo_nivel
        nivel["QTD_SOLICITADA"] = nivel["QTD_PENDENTE_ANTES"] / qtd_alt_mesmo_nivel
        nivel["HR_SOLICITADA"] = nivel["QTD_SOLICITADA"] / nivel["PCS_HORA"]

        nivel["CAP_RESTANTE_ALOC_REC_ANTES"] = nivel["ALOC_REC"].map(cap_restante_rec).fillna(0)
        nivel["CAP_RESTANTE_COD_FER_ANTES"] = nivel["COD_FER_UNID"].map(cap_restante_fer).fillna(0)

        demanda_rec = nivel.groupby("ALOC_REC")["HR_SOLICITADA"].transform("sum")
        demanda_fer = nivel.groupby("COD_FER_UNID")["HR_SOLICITADA"].transform("sum")
        nivel["HR_DEMANDA_TOTAL_ALOC_REC"] = demanda_rec
        nivel["HR_DEMANDA_TOTAL_COD_FER"] = demanda_fer
        nivel["FATOR_RATEIO_ALOC_REC"] = np.where(demanda_rec > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_ALOC_REC_ANTES"] / demanda_rec), 0.0)
        nivel["FATOR_RATEIO_COD_FER"] = np.where(demanda_fer > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_COD_FER_ANTES"] / demanda_fer), 0.0)
        nivel["FATOR_RATEIO"] = nivel[["FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER"]].min(axis=1).clip(lower=0, upper=1)
        nivel["CAP_DISPONIVEL_LINHA"] = nivel[["CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES"]].min(axis=1)

        cond_rec = nivel["FATOR_RATEIO_ALOC_REC"] < nivel["FATOR_RATEIO_COD_FER"] - TOL
        cond_fer = nivel["FATOR_RATEIO_COD_FER"] < nivel["FATOR_RATEIO_ALOC_REC"] - TOL
        cond_ambos = (~cond_rec) & (~cond_fer) & (nivel["FATOR_RATEIO"] < 1 - TOL)
        nivel["MOTIVO_LIMITANTE"] = np.select([cond_rec, cond_fer, cond_ambos], ["ALOC_REC", "COD_FER_UNID", "ALOC_REC_E_COD_FER"], default="SEM_LIMITACAO")

        nivel["HR_PRODUZIR_SOLVER"] = nivel["HR_SOLICITADA"] * nivel["FATOR_RATEIO"]
        nivel["QTD_PRODUZIR_SOLVER"] = nivel["HR_PRODUZIR_SOLVER"] * nivel["PCS_HORA"]
        nivel["QTD_PENDENTE_DEPOIS"] = (nivel["QTD_PENDENTE_ANTES"] - nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].transform("sum")).clip(lower=0)

        nivel["STATUS_DECISAO"] = np.select(
            [nivel["CAP_RESTANTE_ALOC_REC_ANTES"] <= TOL, nivel["CAP_RESTANTE_COD_FER_ANTES"] <= TOL, nivel["HR_PRODUZIR_SOLVER"] <= TOL, nivel["FATOR_RATEIO"] >= 1 - TOL, nivel["FATOR_RATEIO"] < 1 - TOL],
            ["NAO_ALOCADO_CAP_RECURSO_ZERO", "NAO_ALOCADO_CAP_FERRAMENTA_ZERO", "NAO_ALOCADO_RATEIO_ZERO", "ALOCADO_TOTAL_RODADA", "ALOCADO_PARCIAL_RATEIO"],
            default="VERIFICAR"
        )
        nivel["MOTIVO_CORTE"] = np.where(nivel["FATOR_RATEIO"] >= 1 - TOL, "SEM_CORTE", "CORTE_POR_" + nivel["MOTIVO_LIMITANTE"].astype(str))

        for aloc_rec, consumo in nivel.groupby("ALOC_REC")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_rec[aloc_rec] = max(0.0, cap_restante_rec.get(aloc_rec, 0.0) - float(consumo))
        for cod_fer, consumo in nivel.groupby("COD_FER_UNID")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_fer[cod_fer] = max(0.0, cap_restante_fer.get(cod_fer, 0.0) - float(consumo))
        for item, qtd_prod in nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].sum().items():
            pendente[item] = max(0.0, pendente.get(item, 0.0) - float(qtd_prod))

        producao.append(nivel[nivel["QTD_PRODUZIR_SOLVER"] > TOL].copy())
        rastro.append(nivel.copy())

    bd_producao = pd.concat(producao, ignore_index=True) if producao else pd.DataFrame()
    bd_rastro = pd.concat(rastro, ignore_index=True) if rastro else pd.DataFrame()
    return bd_producao, bd_rastro, pendente, cap_restante_rec, cap_restante_fer

bd_plano_producao, rastro_rateio, pendente_final, cap_restante_rec_final, cap_restante_fer_final = executar_rateio_proporcional_dupla_restricao(bd_demanda_solver, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas)
print("Linhas plano produção:", len(bd_plano_producao))
print("Linhas rastro rateio:", len(rastro_rateio))


In [ ]:

# ============================================================
# 05. SAÍDAS DE NEGÓCIO E RASTREABILIDADE
# ============================================================


cols_plano = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "PESO_PROD_KG", "VOL_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO"]
campos_produto_saida = ["COD_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]
campos_produto_saida = [c for c in campos_produto_saida if c in bd_mes.columns]
if campos_produto_saida and "COD_PROD" in campos_produto_saida:
    cadastro_produto_saida = bd_mes[campos_produto_saida].drop_duplicates("COD_PROD")
    for c in ["LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]:
        if c in bd_plano_producao.columns:
            bd_plano_producao = bd_plano_producao.drop(columns=[c])
    bd_plano_producao = bd_plano_producao.merge(cadastro_produto_saida, on="COD_PROD", how="left")

bd_plano_producao["PESO_PROD_KG"] = to_num(bd_plano_producao["PESO_PROD_KG"]) if "PESO_PROD_KG" in bd_plano_producao.columns else 0
bd_plano_producao["VOL_PRODUZIR_SOLVER"] = bd_plano_producao["QTD_PRODUZIR_SOLVER"] * bd_plano_producao["PESO_PROD_KG"] if "QTD_PRODUZIR_SOLVER" in bd_plano_producao.columns else 0
plano_producao = bd_plano_producao[[c for c in cols_plano if c in bd_plano_producao.columns]].copy()

atendido_item = plano_producao.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "QTD_ATENDIDA_SOLVER"])
nao_atend_item = bd_demanda_solver.merge(atendido_item, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
nao_atend_item["QTD_NAO_ATEND_SOLVER"] = (nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(lower=0)
nao_atend_item["PERC_ATENDIDO_SOLVER"] = np.where(nao_atend_item["NEC_PCS"] > 0, nao_atend_item["QTD_ATENDIDA_SOLVER"] / nao_atend_item["NEC_PCS"], 0)
nao_atend_item["PERC_NAO_ATENDIDO_SOLVER"] = 1 - nao_atend_item["PERC_ATENDIDO_SOLVER"]
cols_nao_atend = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER", "PERC_ATENDIDO_SOLVER", "PERC_NAO_ATENDIDO_SOLVER"]
nao_atend_item = nao_atend_item[[c for c in cols_nao_atend if c in nao_atend_item.columns]].copy()

prod_recurso = plano_producao.groupby("ALOC_REC", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ALOC_REC", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"])
resumo_recursos = bd_cap_recursos.merge(prod_recurso, on="ALOC_REC", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0})
resumo_recursos["OCUPACAO_RECURSO_PCT"] = np.where(resumo_recursos["HOR_REC"] > 0, resumo_recursos["HR_PRODUZIR_SOLVER"] / resumo_recursos["HOR_REC"], 0)
resumo_recursos["HR_RECURSO_OCIOSA"] = resumo_recursos["HOR_REC"] - resumo_recursos["HR_PRODUZIR_SOLVER"]
resumo_recursos["ESTOURO_HR_RECURSO"] = np.where(resumo_recursos["HR_PRODUZIR_SOLVER"] > resumo_recursos["HOR_REC"] + TOL, resumo_recursos["HR_PRODUZIR_SOLVER"] - resumo_recursos["HOR_REC"], 0)
resumo_recursos = resumo_recursos[["ALOC_REC", "HOR_REC", "HR_PRODUZIR_SOLVER", "OCUPACAO_RECURSO_PCT", "HR_RECURSO_OCIOSA", "ESTOURO_HR_RECURSO", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"]].sort_values(["OCUPACAO_RECURSO_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

prod_ferramenta = plano_producao.groupby("COD_FER_UNID", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique"), QTD_RECURSOS=("ALOC_REC", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["COD_FER_UNID", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"])
resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on="COD_FER_UNID", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0, "QTD_RECURSOS": 0})
resumo_ferramentas["OCUPACAO_FER_PCT"] = np.where(resumo_ferramentas["HOR_FER"] > 0, resumo_ferramentas["HR_PRODUZIR_SOLVER"] / resumo_ferramentas["HOR_FER"], 0)
resumo_ferramentas["HR_FER_OCIOSA"] = resumo_ferramentas["HOR_FER"] - resumo_ferramentas["HR_PRODUZIR_SOLVER"]
resumo_ferramentas["ESTOURO_HR_FER"] = np.where(resumo_ferramentas["HR_PRODUZIR_SOLVER"] > resumo_ferramentas["HOR_FER"] + TOL, resumo_ferramentas["HR_PRODUZIR_SOLVER"] - resumo_ferramentas["HOR_FER"], 0)
resumo_ferramentas = resumo_ferramentas[["COD_FER_UNID", "HOR_FER", "HR_PRODUZIR_SOLVER", "OCUPACAO_FER_PCT", "HR_FER_OCIOSA", "ESTOURO_HR_FER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"]].sort_values(["OCUPACAO_FER_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

cols_auditoria = ["MES_REF", "ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP"]
auditoria_capacidade = bd_alternativas_item[[c for c in cols_auditoria if c in bd_alternativas_item.columns]].drop_duplicates().sort_values(["ALOC_REC", "COD_FER_UNID"]).copy()

cols_alt = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "ALTERNATIVA_VALIDA", "MOTIVO_INVALIDA"]
alternativas_item = bd_alternativas_item[[c for c in cols_alt if c in bd_alternativas_item.columns]].sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]).copy()

cols_rastro = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO"]
rastro_rateio = rastro_rateio[[c for c in cols_rastro if c in rastro_rateio.columns]].copy() if not rastro_rateio.empty else pd.DataFrame(columns=cols_rastro)

qtd_alt_audit = bd_alternativas_item.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_AUDITADAS=("ALOC_REC", "count"))
qtd_alt_valid = bd_alternativas_validas.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_VALIDAS=("ALOC_REC", "count"))
qtd_alt_usadas = plano_producao.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_USADAS=("ALOC_REC", "count"), QTD_ALOC_REC_USADOS=("ALOC_REC", "nunique"), QTD_COD_FER_USADOS=("COD_FER_UNID", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS"])
qtd_rodadas = rastro_rateio.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_RODADAS_TENTADAS=("RODADA", "nunique"), QTD_ALOC_REC_TENTADOS=("ALOC_REC", "nunique"), QTD_COD_FER_TENTADOS=("COD_FER_UNID", "nunique")) if not rastro_rateio.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"])

diagnostico_item = nao_atend_item.copy()
for df_merge in [qtd_alt_audit, qtd_alt_valid, qtd_alt_usadas, qtd_rodadas]:
    diagnostico_item = diagnostico_item.merge(df_merge, on="ID_PROD_UNID_FAT", how="left")
for c in ["QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"]:
    if c in diagnostico_item.columns:
        diagnostico_item[c] = diagnostico_item[c].fillna(0).astype(int)
diagnostico_item["MOTIVO_SALDO_FINAL"] = np.select([diagnostico_item["QTD_NAO_ATEND_SOLVER"] <= TOL, diagnostico_item.get("QTD_ALTERNATIVAS_AUDITADAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_VALIDAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_USADAS", 0).eq(0)], ["ATENDIDO_TOTAL", "SEM_ALTERNATIVA_CADASTRADA", "SEM_ALTERNATIVA_VALIDA", "SEM_ALOCACAO_COM_ALTERNATIVA_VALIDA"], default="SALDO_APOS_RATEIO_E_LIMITES_DE_CAPACIDADE")

uso_alt = plano_producao.groupby(["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER"])
diagnostico_alternativas = alternativas_item.merge(uso_alt, on=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], how="left")
diagnostico_alternativas["HR_PRODUZIR_SOLVER"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"].fillna(0)
diagnostico_alternativas["QTD_PRODUZIR_SOLVER"] = diagnostico_alternativas["QTD_PRODUZIR_SOLVER"].fillna(0)
diagnostico_alternativas["ALTERNATIVA_USADA"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"] > TOL

print("Resumo recursos:", len(resumo_recursos))
print("Resumo ferramentas:", len(resumo_ferramentas))
print("Plano produção:", len(plano_producao))
print("Não atendimento:", len(nao_atend_item))
print("Rastro:", len(rastro_rateio))


In [ ]:

# ============================================================
# 06. VALIDAÇÕES
# ============================================================

nec_total = nao_atend_item["NEC_PCS"].sum()
atend_total = nao_atend_item["QTD_ATENDIDA_SOLVER"].sum()
nao_atend_total = nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum()
dif_fechamento = nec_total - atend_total - nao_atend_total
qtd_rec_estouro = (resumo_recursos["ESTOURO_HR_RECURSO"] > TOL).sum()
qtd_fer_estouro = (resumo_ferramentas["ESTOURO_HR_FER"] > TOL).sum()

validacao = pd.DataFrame({
    "METRICA": ["NEC_PCS_TOTAL", "QTD_ATENDIDA_SOLVER_TOTAL", "QTD_NAO_ATEND_SOLVER_TOTAL", "DIF_FECHAMENTO_DEMANDA", "QTD_ITENS_DEMANDA", "QTD_ITENS_COM_PRODUCAO", "QTD_ITENS_SEM_PRODUCAO", "QTD_ITENS_COM_NAO_ATENDIMENTO", "QTD_RECURSOS", "QTD_RECURSOS_ESTOURO", "ESTOURO_HR_RECURSO_TOTAL", "HOR_REC_TOTAL", "HR_PRODUZIR_SOLVER_RECURSO_TOTAL", "OCUPACAO_GLOBAL_RECURSOS", "QTD_FERRAMENTAS", "QTD_FERRAMENTAS_ESTOURO", "ESTOURO_HR_FER_TOTAL", "HOR_FER_TOTAL", "HR_PRODUZIR_SOLVER_FER_TOTAL", "OCUPACAO_GLOBAL_FERRAMENTAS", "QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS_PRODUCAO", "QTD_ALTERNATIVAS_HOR_CAP_ZERO", "QTD_LINHAS_RASTRO_RATEIO", "QTD_ITENS_DIAGNOSTICO"],
    "VALOR": [nec_total, atend_total, nao_atend_total, dif_fechamento, len(nao_atend_item), (nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL).sum(), (nao_atend_item["QTD_ATENDIDA_SOLVER"] <= TOL).sum(), (nao_atend_item["QTD_NAO_ATEND_SOLVER"] > TOL).sum(), len(resumo_recursos), qtd_rec_estouro, resumo_recursos["ESTOURO_HR_RECURSO"].sum(), resumo_recursos["HOR_REC"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum() / resumo_recursos["HOR_REC"].sum() if resumo_recursos["HOR_REC"].sum() > 0 else 0, len(resumo_ferramentas), qtd_fer_estouro, resumo_ferramentas["ESTOURO_HR_FER"].sum(), resumo_ferramentas["HOR_FER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum() / resumo_ferramentas["HOR_FER"].sum() if resumo_ferramentas["HOR_FER"].sum() > 0 else 0, len(bd_alternativas_item), len(bd_alternativas_validas), (bd_alternativas_item["HOR_CAP"] <= TOL).sum(), len(rastro_rateio), len(diagnostico_item)]
})
validacao_checks = pd.DataFrame({"METRICA": ["FECHAMENTO_DEMANDA_OK", "CAPACIDADE_RECURSOS_OK", "CAPACIDADE_FERRAMENTAS_OK"], "VALOR": [abs(dif_fechamento) <= 1e-4, qtd_rec_estouro == 0, qtd_fer_estouro == 0]})
validacao = pd.concat([validacao, pd.DataFrame({"METRICA": ["---"], "VALOR": [""]}), validacao_checks], ignore_index=True)
display(validacao)


In [ ]:

# ============================================================
# 07. EXPORTAÇÃO
# ============================================================

OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    resumo_recursos.to_excel(writer, sheet_name="01_RESUMO_RECURSOS", index=False)
    resumo_ferramentas.to_excel(writer, sheet_name="02_RESUMO_FERRAMENTAS", index=False)
    plano_producao.to_excel(writer, sheet_name="03_PLANO_PRODUCAO", index=False)
    nao_atend_item.to_excel(writer, sheet_name="04_NAO_ATEND_ITEM", index=False)
    auditoria_capacidade.to_excel(writer, sheet_name="05_AUDITORIA_CAPACIDADE", index=False)
    alternativas_item.to_excel(writer, sheet_name="06_ALTERNATIVAS_ITEM", index=False)
    rastro_rateio.to_excel(writer, sheet_name="07_RASTRO_RATEIO", index=False)
    diagnostico_item.to_excel(writer, sheet_name="08_DIAGNOSTICO_ITEM", index=False)
    diagnostico_alternativas.to_excel(writer, sheet_name="09_DIAGNOSTICO_ALTERNATIVAS", index=False)
    validacao.to_excel(writer, sheet_name="99_VALIDACAO", index=False)
formatar_excel(OUTPUT_SOLVER_FILE)
print("Arquivo exportado:", OUTPUT_SOLVER_FILE)
wb = load_workbook(OUTPUT_SOLVER_FILE, read_only=True)
print("Guias exportadas:")
for s in wb.sheetnames:
    print("-", s)


In [ ]:
timer.finalizar()
